# 02 数据清洗与存储

本 Notebook 完成第二部分（进阶存储）和第三部分（数据清洗）的全部任务：

- **3.1** 单表清洗：缺失值检测与处理、日期格式统一、数据类型检查、重复值处理、离群值标注
- **3.2** 宽表与长表转换
- **3.3** 多表合并（个股 + 指数 + 宏观）
- **2.2 方式 B** Parquet 格式演示与 CSV 对比

## 0. 环境准备

In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

STOCKS = {
    '603685': '晨丰科技', '603319': '美湖股份', '600519': '贵州茅台',
    '601288': '农业银行', '601166': '兴业银行', '600048': '保利发展',
    '000568': '泸州老窖', '002179': '中航光电', '300510': '金冠股份', '000988': '华工科技',
}

print("环境准备完毕")

环境准备完毕


---
# 第三部分：数据清洗

## 3.1 单表清洗

对每只股票的原始 CSV 进行以下清洗：缺失值检测→处理→日期格式统一→数据类型检查→重复值处理→离群值标注。

### 3.1.1 缺失值检测

首先统计所有股票原始数据中每列的缺失值数量和比例，制成汇总表格，并分析可能原因。

In [2]:
missing_summary = []

for code, name in STOCKS.items():
    df = pd.read_csv(f'data/stock/stock_{code}.csv', encoding='utf-8-sig')
    for col in df.columns:
        n_miss = df[col].isna().sum()
        pct = n_miss / len(df) * 100
        missing_summary.append({
            '股票代码': code, '股票名称': name,
            '列名': col, '缺失数量': n_miss, '缺失比例(%)': round(pct, 2)
        })

df_miss = pd.DataFrame(missing_summary)
# 只展示有缺失的
df_miss_nonzero = df_miss[df_miss['缺失数量'] > 0]
if len(df_miss_nonzero) == 0:
    print("✅ 所有股票原始数据均无缺失值")
else:
    print(df_miss_nonzero.to_string(index=False))
print(f"\n总检测列数: {len(df_miss)}")

✅ 所有股票原始数据均无缺失值

总检测列数: 80


**清洗说明**：检测结果显示各股票行情数据缺失值极少或为零，这与数据来源特性一致——后复权日度行情已经过数据提供方预处理。若存在少量缺失，可能来自以下原因：
1. **停牌**：股票临时停牌期间无成交数据，部分字段可能为空；
2. **节假日/非交易日**：行情数据仅包含交易日，但日期不连续不等于缺失；
3. **数据源同步延迟**：接口返回时极少数最新日期数据尚未入库。

处理策略：对价格类缺失使用**向前填充（ffill）**——即沿用停牌前最后一个有效价格，这符合「停牌期间价格不变」的经济含义，优于删除（删除会破坏时间序列的连续性，影响后续收益率计算）。

### 3.1.2–3.1.5 完整清洗流程（每只股票）

In [3]:
clean_records = []  # 记录清洗统计信息
dfs_clean = {}      # 存储清洗后的 DataFrame

for code, name in STOCKS.items():
    df = pd.read_csv(f'data/stock/stock_{code}.csv', encoding='utf-8-sig')
    n0 = len(df)

    # --- 3.1.3 日期格式统一 ---
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()

    # --- 3.1.4 数据类型检查 ---
    numeric_cols = ['open', 'close', 'high', 'low', 'volume', 'amount']
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # --- 3.1.2 缺失值处理：向前填充 ---
    n_miss_before = df.isna().sum().sum()
    df = df.ffill()
    n_miss_after = df.isna().sum().sum()

    # --- 3.1.5 重复值处理 ---
    n_dup = df.index.duplicated().sum()
    df = df[~df.index.duplicated(keep='first')]

    # --- 3.1.6 离群值标注 ---
    df['return'] = np.log(df['close'] / df['close'].shift(1))
    df['is_extreme'] = df['return'].abs() > 0.20  # 单日涨跌幅超过±20%
    n_extreme = df['is_extreme'].sum()

    dfs_clean[code] = df
    clean_records.append({
        '代码': code, '名称': name, '原始行数': n0,
        '清洗后行数': len(df), '删除重复': n_dup,
        '填充缺失': n_miss_before, '极端收益日': int(n_extreme)
    })

df_clean_stat = pd.DataFrame(clean_records)
print(df_clean_stat.to_string(index=False))

    代码   名称  原始行数  清洗后行数  删除重复  填充缺失  极端收益日
603685 晨丰科技  1543   1543     0     0      0
603319 美湖股份  1545   1545     0     0      0
600519 贵州茅台  1545   1545     0     0      0
601288 农业银行  1545   1545     0     0      0
601166 兴业银行  1545   1545     0     0      0
600048 保利发展  1545   1545     0     0      0
000568 泸州老窖  1545   1545     0     0      0
002179 中航光电  1545   1545     0     0      0
300510 金冠股份  1545   1545     0     0      1
000988 华工科技  1545   1545     0     0      0


**清洗说明**：

- **日期列**：已从字符串转换为 `datetime64` 格式并设为索引，便于时间序列操作（resample、shift、rolling 等）。
- **数值列**：open/close/high/low/volume/amount 均已确认为 float64 类型，无需额外转换。
- **重复值**：若同一日期出现多条记录，保留第一条并删除其余，以避免重复计算影响统计结果。
- **离群值（is_extreme）**：单日对数收益率超过 ±20% 的记录标注为 True，**不删除**，保留原始数据完整性。这类极端值可能的成因包括：股票复牌、送转股导致价格跳升、市场异常波动等。在后续分析中可根据需要选择是否纳入计算。

### 3.1.6 离群值分布展示

In [4]:
print("各股票极端收益日（|return| > 20%）明细：")
for code, name in STOCKS.items():
    df = dfs_clean[code]
    extremes = df[df['is_extreme'] == True]
    if len(extremes) > 0:
        print(f"  {code}({name}): {len(extremes)} 天")
        for idx, row in extremes.iterrows():
            print(f"    {idx.date()}  return={row['return']:.2%}  close={row['close']:.2f}")
    else:
        print(f"  {code}({name}): 无极端收益日")

各股票极端收益日（|return| > 20%）明细：
  603685(晨丰科技): 无极端收益日
  603319(美湖股份): 无极端收益日
  600519(贵州茅台): 无极端收益日
  601288(农业银行): 无极端收益日
  601166(兴业银行): 无极端收益日
  600048(保利发展): 无极端收益日
  000568(泸州老窖): 无极端收益日
  002179(中航光电): 无极端收益日
  300510(金冠股份): 1 天
    2025-04-07  return=-20.27%  close=22.16
  000988(华工科技): 无极端收益日


---
## 3.2 宽表与长表转换

将 10 只股票的收盘价合并为**宽表**（日期为索引，每列一只股票），再用 `pd.melt()` 转换回**长表**。

In [5]:
# 宽表：日期为索引，每列为一只股票的收盘价
wide_close = pd.DataFrame({
    code: dfs_clean[code]['close']
    for code in STOCKS
})
wide_close.index.name = 'date'

print(f"宽表形状: {wide_close.shape}")
print("宽表前 3 行:")
print(wide_close.head(3).to_string())

宽表形状: (1545, 10)
宽表前 3 行:
            603685  603319   600519  601288  601166  600048   000568  002179  300510  000988
date                                                                                        
2020-01-02   23.47   24.28  8495.30    5.83   83.36  333.51  2767.55  305.83   44.13  209.73
2020-01-03   23.40   24.36  8108.58    5.82   83.07  327.16  2778.56  308.47   45.62  210.23
2020-01-06   23.27   24.41  8104.29    5.77   82.12  321.62  2773.05  313.66   44.50  220.69


In [6]:
# 长表：用 pd.melt 转换，字段为 date, code, close
long_close = wide_close.reset_index().melt(
    id_vars='date', var_name='code', value_name='close'
)
long_close = long_close.sort_values(['date','code']).reset_index(drop=True)

print(f"长表形状: {long_close.shape}")
print("长表前 5 行:")
print(long_close.head(5))

长表形状: (15450, 3)
长表前 5 行:
        date    code    close
0 2020-01-02  000568  2767.55
1 2020-01-02  000988   209.73
2 2020-01-02  002179   305.83
3 2020-01-02  300510    44.13
4 2020-01-02  600048   333.51


**宽表 vs 长表的适用场景：**

- **宽表**（每列一只股票）适合：计算相关系数矩阵（`df.corr()`）、绘制多股票对比走势图、矩阵运算（协方差矩阵估计）。宽表的优势是列操作直观，横向对齐天然。

- **长表**（每行一条记录）适合：分组统计（`groupby('code').agg()`）、分面可视化（seaborn `FacetGrid`）、与其他数据表做 `merge`/`join`、存入数据库（每行对应一条事实记录）。长表遵循「整洁数据（Tidy Data）」原则，更适合大多数数据分析框架。

---
## 3.3 多表合并

### 3.3.1 个股数据合并（10 只股票纵向堆叠）

In [7]:
# 纵向合并所有个股清洗数据
all_stocks = []
for code, name in STOCKS.items():
    df = dfs_clean[code].copy().reset_index()
    df['code'] = code
    df['name'] = name
    all_stocks.append(df)

df_all = pd.concat(all_stocks, ignore_index=True)
print(f"合并前（每只股票独立）: 10 个 DataFrame，各约 1545 行")
print(f"合并后（纵向堆叠）: {df_all.shape}")
print(f"行数变化说明: 10 只股票 × 约 1545 个交易日 = {len(df_all)} 行")

合并前（每只股票独立）: 10 个 DataFrame，各约 1545 行
合并后（纵向堆叠）: (15448, 12)
行数变化说明: 10 只股票 × 约 1545 个交易日 = 15448 行


### 3.3.2 个股 + 指数：按日期 left join

In [8]:
# 读取沪深300指数
df_idx = pd.read_csv('data/index/index_000300.csv', encoding='utf-8-sig')
df_idx['date'] = pd.to_datetime(df_idx['date'])
df_idx = df_idx.rename(columns={'close': 'hs300_close', 'open': 'hs300_open'})
keep_idx = ['date'] + [c for c in df_idx.columns if 'hs300' in c]
df_idx = df_idx[keep_idx]

n_before = len(df_all)
df_with_idx = df_all.merge(df_idx, on='date', how='left')
n_after = len(df_with_idx)

print(f"合并前行数: {n_before}")
print(f"合并后行数: {n_after}（相等，因为 left join 保留所有个股记录）")
print(f"行数变化: {n_after - n_before}")
print(f"沪深300收盘价缺失（无对应交易日）: {df_with_idx['hs300_close'].isna().sum()} 行")
print("说明：个股数据与指数数据均为交易日，left join 后行数不变；")
print("少量 NaN 来自两者交易日不完全重叠（如某些特殊停牌日）。")

合并前行数: 15448
合并后行数: 15448（相等，因为 left join 保留所有个股记录）
行数变化: 0
沪深300收盘价缺失（无对应交易日）: 0 行
说明：个股数据与指数数据均为交易日，left join 后行数不变；
少量 NaN 来自两者交易日不完全重叠（如某些特殊停牌日）。


### 3.3.3 日度数据 + 月度宏观数据（频率对齐）

宏观数据（CPI、汇率）为月度频率，个股数据为日度频率。处理策略：将宏观数据的月度值**向前填充**至该月每个交易日，即每个交易日使用当月对应的宏观指标值。

In [9]:
# 读取宏观数据
df_fx = pd.read_csv('data/macro/macro_exchange_rate.csv', encoding='utf-8-sig')
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx['year_month'] = df_fx['date'].dt.to_period('M')

# 给日度数据加上年月键
df_with_idx['year_month'] = df_with_idx['date'].dt.to_period('M')

n_before2 = len(df_with_idx)
df_combined = df_with_idx.merge(
    df_fx[['year_month', 'usd_cny_mid']],
    on='year_month', how='left'
)
n_after2 = len(df_combined)

print(f"合并前行数: {n_before2}")
print(f"合并后行数: {n_after2}")
print(f"汇率缺失行数: {df_combined['usd_cny_mid'].isna().sum()}")
print("说明：月度宏观数据通过 year_month 键映射到日度数据，")
print("同一个月内所有交易日使用相同的月均汇率值。")
print("行数不变，因为 left join 保留所有日度记录。")
print("少量 NaN 来自汇率数据时间范围与股票数据不完全重叠。")

df_combined = df_combined.drop(columns=['year_month'])
print(f"\n综合数据字段: {list(df_combined.columns)}")

合并前行数: 15448
合并后行数: 15448
汇率缺失行数: 0
说明：月度宏观数据通过 year_month 键映射到日度数据，
同一个月内所有交易日使用相同的月均汇率值。
行数不变，因为 left join 保留所有日度记录。
少量 NaN 来自汇率数据时间范围与股票数据不完全重叠。

综合数据字段: ['date', 'open', 'close', 'high', 'low', 'volume', 'amount', 'amount.1', 'return', 'is_extreme', 'code', 'name', 'hs300_open', 'hs300_close', 'usd_cny_mid']


### 保存合并后综合数据

In [10]:
# 保存至 data/combined/
df_combined.to_csv('data/combined/combined_data.csv', index=False, encoding='utf-8-sig')
print(f"综合数据已保存: data/combined/combined_data.csv  shape={df_combined.shape}")

# 同时保存个股清洗数据（方式 A CSV）
# 构建 stock_clean: 纵向堆叠，含所有清洗字段
df_stock_clean = df_all.copy()
df_stock_clean.to_csv('data/clean/stock_clean.csv', index=False, encoding='utf-8-sig')
print(f"清洗后股票数据已保存: data/clean/stock_clean.csv  shape={df_stock_clean.shape}")

综合数据已保存: data/combined/combined_data.csv  shape=(15448, 15)


清洗后股票数据已保存: data/clean/stock_clean.csv  shape=(15448, 12)


---
# 第二部分 2.2 方式 B：Parquet 格式演示

在完成 CSV（方式 A）存储后，额外将清洗数据保存为 Parquet 格式，并与 CSV 进行对比。

In [11]:
import pandas as pd
import pyarrow.parquet as pq
import os
import time

# 将清洗后数据保存为 Parquet
parquet_path = 'data/clean/stock_clean.parquet'
df_stock_clean.to_parquet(parquet_path, index=False, engine='pyarrow')
print(f"Parquet 文件已保存: {parquet_path}")

Parquet 文件已保存: data/clean/stock_clean.parquet


In [12]:
# 列式读取（只加载需要的列）
df_partial = pd.read_parquet('data/clean/stock_clean.parquet',
                              columns=['date', 'code', 'close'])
print("列式读取（仅 date/code/close）:")
print(df_partial.head(3))
print(f"shape: {df_partial.shape}")

列式读取（仅 date/code/close）:
        date    code  close
0 2020-01-02  603685  23.47
1 2020-01-03  603685  23.40
2 2020-01-06  603685  23.27
shape: (15448, 3)


In [13]:
# 查看 Schema（类型契约）
schema = pq.read_schema('data/clean/stock_clean.parquet')
print("Parquet Schema（数据类型契约）:")
print(schema)

Parquet Schema（数据类型契约）:
date: timestamp[us]
open: double
close: double
high: double
low: double
volume: double
amount: double
amount.1: double
return: double
is_extreme: bool
code: large_string
name: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1425


In [14]:
# 与 CSV 对比：读取速度与文件体积
csv_path = 'data/clean/stock_clean.csv'
parquet_path = 'data/clean/stock_clean.parquet'

# CSV 读取速度
t0 = time.time()
_ = pd.read_csv(csv_path)
csv_time = time.time() - t0
csv_size = os.path.getsize(csv_path) / 1024

# Parquet 读取速度
t0 = time.time()
_ = pd.read_parquet(parquet_path)
parquet_time = time.time() - t0
parquet_size = os.path.getsize(parquet_path) / 1024

print(f"CSV     读取耗时: {csv_time:.3f}s   文件大小: {csv_size:.1f} KB")
print(f"Parquet 读取耗时: {parquet_time:.3f}s   文件大小: {parquet_size:.1f} KB")
print(f"\n体积压缩比: {csv_size/parquet_size:.1f}x（CSV 是 Parquet 的 {csv_size/parquet_size:.1f} 倍）")
print(f"速度比: CSV 耗时是 Parquet 的 {csv_time/max(parquet_time,0.001):.1f} 倍")

CSV     读取耗时: 0.037s   文件大小: 1839.0 KB
Parquet 读取耗时: 0.005s   文件大小: 716.4 KB

体积压缩比: 2.6x（CSV 是 Parquet 的 2.6 倍）
速度比: CSV 耗时是 Parquet 的 7.5 倍


**对比分析与讨论：**

在本次数据规模下（约 1.5 万行 × 10 只股票 = 约 15,000 行），CSV 和 Parquet 在读取速度上的差异**不明显**——两者均在毫秒到几十毫秒级别，人感知不到差别。文件体积方面，Parquet 通常比 CSV 小 **30%–50%**，在本项目规模下节省的空间有限（几十至几百 KB），实际体验差异不显著。

**差异会在以下场景变得显著：**
1. **数据量达到百万行以上**（如全市场 5000 只股票 × 5 年日度数据）：Parquet 读取速度可快 5–20 倍，体积压缩比达 3–10 倍；
2. **只需读取部分列**（如仅读 `date, close`）：Parquet 的列式存储跳过不需要的列，而 CSV 必须解析整行；
3. **数据类型复杂时**（如含 datetime、bool 混合类型）：CSV 读入后需手动 `parse_dates`，Parquet Schema 自动恢复类型，工程效率更高。

## 清洗结果总览

In [15]:
print("="*60)
print("清洗后文件清单:")
for dirpath in ['data/clean', 'data/combined']:
    for f in sorted(os.listdir(dirpath)):
        p = os.path.join(dirpath, f)
        size = os.path.getsize(p) / 1024
        print(f"  {p:<45} {size:.1f} KB")

print("\n清洗统计汇总:")
print(df_clean_stat.to_string(index=False))

清洗后文件清单:
  data/clean\stock_clean.csv                    1839.0 KB
  data/clean\stock_clean.parquet                716.4 KB
  data/combined\combined_data.csv               2314.8 KB

清洗统计汇总:
    代码   名称  原始行数  清洗后行数  删除重复  填充缺失  极端收益日
603685 晨丰科技  1543   1543     0     0      0
603319 美湖股份  1545   1545     0     0      0
600519 贵州茅台  1545   1545     0     0      0
601288 农业银行  1545   1545     0     0      0
601166 兴业银行  1545   1545     0     0      0
600048 保利发展  1545   1545     0     0      0
000568 泸州老窖  1545   1545     0     0      0
002179 中航光电  1545   1545     0     0      0
300510 金冠股份  1545   1545     0     0      1
000988 华工科技  1545   1545     0     0      0
